In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
just_checking_integrity=False

In [ ]:
import gc
import pandas as pd
import scipy.sparse as sps

In [ ]:
valid=sps.load_npz("/content/drive/MyDrive/RS2024/Trial02/recsys2022/dataset/processed_data/URM_valid_bought.npz")
bought=valid.indices

In [ ]:
dataset=None
dataset_max=None
#["UCF","ICBF","ICF","Graph","TopPop100","TopPop50","TopPop15"]
#for file in ["UCF","ICBF","Graph","ICF","UCF_W","ICBF_W","Graph_W","ICF_W","UCF_WW","ICBF_WW","Graph_WW","ICF_WW"]:
for file in ["UCF_W"]:
    print(gc.collect())
    print(f"loading {file}")
    df=pd.read_csv(f"/content/drive/MyDrive/RS2024/Trial02/recsys2022/dataset/candidates/traditional_recs/train/{file}.csv")
    print(len(df))
    print(f"loaded {file}")
    for col in df:
        if "Max" in col:
            df_max=df[["Session_Id",col]].drop_duplicates()
        elif "Score" in col:
            df_feats=df[["Session_Id","Item_ID",col]]
    if dataset is None:
        dataset=df_feats
        dataset_max=df_max
    else:
        dataset=dataset.merge(df_feats,on=["Session_Id","Item_ID"],how="outer")
        dataset_max=dataset_max.merge(df_max,on=["Session_Id"],how="outer")
    del df


#del dataset_max
del df_feats
del df_max

0
loading UCF_W
18588455
loaded UCF_W


In [ ]:
import ast
submission_df=pd.read_csv("/content/drive/MyDrive/RS2024/Trial02/recsys2022/dataset/candidates/NN/GRU/train/candidates.csv")
if just_checking_integrity:
    submission_df["score"]=submission_df["score"].apply(lambda x:x.replace("-inf","0")).apply(lambda x:x.replace("+inf","0")).apply(lambda x:x.replace("inf","0"))
    submission_df["score"]=submission_df["score"].map(ast.literal_eval)
else:
    submission_df["score"]=submission_df["score"].map(ast.literal_eval)
submission_df["code"]=submission_df["code"].map(ast.literal_eval)
submission_df["rank"]=submission_df["rank"].map(ast.literal_eval)
submission_df = submission_df.explode(['code','score', 'rank'])
submission_df[["session_id","code","rank"]]=submission_df[["session_id","code","rank"]].astype(int)
submission_df.rename(columns={"code":"Item_ID","session_id":"Session_Id","rank":"rank_gru","score":"score_gru"},inplace=True)

In [ ]:
submission_df.head()

,Session_Id,Item_ID,score_gru,rank_gru
0,918382,3844,0.021967,1
0,918382,3678,0.016598,2
0,918382,1189,0.015711,3
0,918382,446,0.01126,4
0,918382,1813,0.009247,5


In [ ]:
# import ast
# submission_df_tr=pd.read_csv("/content/drive/MyDrive/RS2024/Trial02/recsys2022/dataset/candidates/NN/Transformer/train/candidates.csv")
# if just_checking_integrity:
#     submission_df_tr["score"]=submission_df_tr["score"].apply(lambda x:x.replace("-inf","0"))
#     submission_df_tr["score"]=submission_df_tr["score"].map(ast.literal_eval)
# else:
#     submission_df_tr["score"]=submission_df_tr["score"].map(ast.literal_eval)
# submission_df_tr["code"]=submission_df_tr["code"].map(ast.literal_eval)
# submission_df_tr["rank"]=submission_df_tr["rank"].map(ast.literal_eval)
# submission_df_tr = submission_df_tr.explode(['code','score', 'rank'])
# submission_df_tr[["session_id","code","rank"]]=submission_df_tr[["session_id","code","rank"]].astype(int)
# submission_df_tr.rename(columns={"code":"Item_ID","session_id":"Session_Id","rank":"rank_transformer","score":"score_transformer"},inplace=True)

In [ ]:
# import ast
# submission_df_LSTM=pd.read_csv("/content/drive/MyDrive/RS2024/Trial02/recsys2022/dataset/candidates/NN/LSTM/train/candidates.csv")
# if just_checking_integrity:
#     submission_df_LSTM["score"]=submission_df_LSTM["score"].apply(lambda x:x.replace("-inf","0"))
#     submission_df_LSTM["score"]=submission_df_LSTM["score"].map(ast.literal_eval)
# else:
#     submission_df_LSTM["score"]=submission_df_LSTM["score"].map(ast.literal_eval)
# submission_df_LSTM["code"]=submission_df_LSTM["code"].map(ast.literal_eval)
# submission_df_LSTM["rank"]=submission_df_LSTM["rank"].map(ast.literal_eval)
# submission_df_LSTM = submission_df_LSTM.explode(['code','score', 'rank'])
# submission_df_LSTM[["session_id","code","rank"]]=submission_df_LSTM[["session_id","code","rank"]].astype(int)
# submission_df_LSTM.rename(columns={"code":"Item_ID","session_id":"Session_Id","rank":"rank_LSTM","score":"score_LSTM"},inplace=True)

In [ ]:
dataset=dataset.merge(submission_df,on=["Session_Id","Item_ID"],how="outer")
#dataset=dataset.merge(submission_df_tr,on=["Session_Id","Item_ID"],how="outer")
#dataset=dataset.merge(submission_df_LSTM,on=["Session_Id","Item_ID"],how="outer")

dataset=dataset.merge(dataset_max,on=["Session_Id"],how="left")


In [ ]:
if just_checking_integrity:
    dataset=dataset[:int(0.5*len(dataset))]

In [ ]:
#del dataset_max
#del submission_df
#del submission_df_tr

In [ ]:
import numpy as np
sessions=np.unique(dataset["Session_Id"])
train=pd.DataFrame()
train["Session_Id"]=sessions
if just_checking_integrity:
    train["Item_ID"]=bought[:len(sessions)]
else:
    train["Item_ID"]=bought
train["target"]=1

In [ ]:
gc.collect()

209

In [ ]:
dataset=dataset.merge(train,on=["Session_Id","Item_ID"],how="left")
dataset.fillna(0,inplace=True)
gc.collect()

0

In [ ]:
dataset["target"]=dataset["target"].astype("int8")
gc.collect()

0

In [ ]:
sessions_containing_bought_among_candidates=dataset[["Session_Id","target"]].groupby("Session_Id").agg(sum).reset_index()
sessions_containing_bought_among_candidates=sessions_containing_bought_among_candidates.loc[sessions_containing_bought_among_candidates["target"]>0]

dataset=dataset[dataset['Session_Id'].isin(sessions_containing_bought_among_candidates["Session_Id"].tolist())]

In [ ]:
NHITS=len(dataset['Session_Id'].unique())
HITRATE=NHITS/81600 #approximate value
HITRATE

0.6025

In [ ]:
dataset.head(20)

,Session_Id,Item_ID,Score_UCF_W,score_gru,rank_gru,Max_Score_UCF_W,target
209,918383,55,0.054177,0.000000,0.0,594.02826,0
210,918383,122,0.025767,0.000000,0.0,594.02826,0
211,918383,189,0.129413,0.000446,76.0,594.02826,0
212,918383,215,0.036236,0.000000,0.0,594.02826,0
213,918383,260,0.039900,0.000000,0.0,594.02826,0
214,918383,320,0.040230,0.000786,42.0,594.02826,0
215,918383,357,0.040280,0.000000,0.0,594.02826,0
216,918383,379,0.046153,0.000000,0.0,594.02826,0
217,918383,421,0.064009,0.000369,91.0,594.02826,0
218,918383,506,0.032975,0.000000,0.0,594.02826,0


In [ ]:
dataset

,Session_Id,Item_ID,Score_UCF_W,score_gru,rank_gru,Max_Score_UCF_W,target
209,918383,55,0.054177,0.000000,0.0,594.028260,0
210,918383,122,0.025767,0.000000,0.0,594.028260,0
211,918383,189,0.129413,0.000446,76.0,594.028260,0
212,918383,215,0.036236,0.000000,0.0,594.028260,0
213,918383,260,0.039900,0.000000,0.0,594.028260,0
...,...,...,...,...,...,...,...
22882238,999998,4347,0.000000,0.000617,96.0,81.306076,0
22882239,999998,4587,0.000000,0.000617,97.0,81.306076,0
22882240,999998,1714,0.000000,0.000616,98.0,81.306076,0
22882241,999998,260,0.000000,0.000616,99.0,81.306076,0


In [ ]:
dataset_macro_features = pd.read_csv('/content/drive/MyDrive/RS2024/Trial02/recsys2022/dataset/processed_data/macro_feats_NN.csv',
                                     usecols=['session_id',
                                                'session_similarity',
                                                'session_similarity_uniques', 'session_similarity_feats', 'session_similarity_feats_uniques',
                                              'length_of_session_seconds',
                   'avg_time_spent_per_item_seconds',
                   'variance_time_spent_per_item_seconds',
                   'n_seen_items',
                   'n_unique_items',])

dataset_macro_features.rename(columns={'session_id':'Session_Id'}, inplace=True)
dataset = dataset.merge(dataset_macro_features, on='Session_Id', how='left')
del dataset_macro_features

In [ ]:
#dataset.drop('Max_Score_TopPop15', axis=1, inplace=True)

In [ ]:
from sklearn.model_selection import KFold
import lightgbm
import os
k=10


score=0
kf = KFold(n_splits=k)
index=0
for train_index, valid_index in kf.split(train):

    train_sessions=train["Session_Id"][train_index].tolist()
    valid_sessions=train["Session_Id"][valid_index].tolist()

    train_set=dataset[dataset["Session_Id"].isin(train_sessions)]
    valid_set=dataset[dataset["Session_Id"].isin(valid_sessions)]

    train_set=train_set.sort_values("Session_Id")
    valid_set=valid_set.sort_values("Session_Id")

    qids_train = train_set[["Session_Id"]].groupby("Session_Id")["Session_Id"].count().to_numpy()
    qids_valid = valid_set[["Session_Id"]].groupby("Session_Id")["Session_Id"].count().to_numpy()

    X_train=train_set.drop(["target","Session_Id","Item_ID"],axis=1)
    y_train=train_set["target"]

    X_valid=valid_set.drop(["target","Session_Id","Item_ID"],axis=1)
    y_valid=valid_set["target"]

    model = lightgbm.LGBMRanker(
    objective="lambdarank",
    #objective="rank_xendcg",
    #metric="mrr",
    # n_estimators=2000,
    n_estimators=100,
    lambdarank_truncation_level=103,
    reg_alpha=1e-4,
    learning_rate=0.05,
    min_child_samples=400,
    subsample=0.8,
    subsample_freq=20
    )
    ranker=model.fit(
        X=X_train,
        y=y_train,
        group=qids_train,
        eval_set=[(X_valid, y_valid)],
        eval_group=[qids_valid],
        eval_at=(100),
        callbacks=[lightgbm.log_evaluation(),lightgbm.early_stopping(20,first_metric_only=True)],
        #verbose=10,
        eval_metric=["map","ndcg"]
    )
    ranker.booster_.save_model(f'/content/drive/MyDrive/RS2024/Trial02/recsys2022/rankers/model_{index}.txt')
    #Get maximum val score
    score+=np.max(ranker._evals_result["valid_0"]["map@100"])
    index+=1

[LightGBM] [Info] Total groups: 43740, total data: 12216239
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.560352 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2798
[LightGBM] [Info] Number of data points in the train set: 12216239, number of used features: 13
[LightGBM] [Info] Total groups: 5424, total data: 1522006
[1]	valid_0's map@100: 0.225362	valid_0's ndcg@100: 0.351545
Training until validation scores don't improve for 20 rounds
[2]	valid_0's map@100: 0.255184	valid_0's ndcg@100: 0.374718
[3]	valid_0's map@100: 0.259942	valid_0's ndcg@100: 0.379164
[4]	valid_0's map@100: 0.259876	valid_0's ndcg@100: 0.379016
[5]	valid_0's map@100: 0.260764	valid_0's ndcg@100: 0.379822
[6]	valid_0's map@100: 0.260681	valid_0's ndcg@100: 0.379682
[7]	valid_0's map@100: 0.261224	valid_0's ndcg@100: 0.380157
[8]	valid_0's map@100: 0.261183	valid_

In [ ]:
final_performcance=score/k
print("overall MAP@100 = ", final_performcance)

overall MAP@100 =  0.28914821284175457


In [ ]:
print("normalized score ", final_performcance*HITRATE)

normalized score  0.17421179823715713
